# 랜덤신호론 11주차 실습
## FFT와 전력스펙트럼밀도

1. 시간영역과 주파수영역을 구분한다.
2. FFT에서 지배주파수를 찾는다.
3. PSD를 주파수별 전력 분포로 설명한다.
4. 지배주파수와 스펙트럼 에너지를 AI 특징으로 추출한다.


## 0. 공통 환경 준비


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

rng=np.random.default_rng(2026)
plt.style.use('seaborn-v0_8-whitegrid')
FEATURE_NAMES=['mean','std','peak_to_peak','rms','autocorrelation','dominant_frequency','spectral_energy']

def lag1_autocorrelation(x):
    x=np.asarray(x,float)
    if len(x)<2 or np.std(x[:-1])==0 or np.std(x[1:])==0: return 0.0
    return float(np.corrcoef(x[:-1],x[1:])[0,1])

def extract_features(x,fs):
    x=np.asarray(x,float); centered=x-x.mean()
    spectrum=np.abs(np.fft.rfft(centered)); freqs=np.fft.rfftfreq(len(x),1/fs)
    return {'mean':float(x.mean()),'std':float(x.std()),'peak_to_peak':float(np.ptp(x)),
      'rms':float(np.sqrt(np.mean(x**2))),'autocorrelation':lag1_autocorrelation(x),
      'dominant_frequency':float(freqs[np.argmax(spectrum)]),
      'spectral_energy':float(np.sum(spectrum**2)/len(x))}

def generate_signal(kind,fs=50,seconds=4,seed=0):
    local=np.random.default_rng(seed); t=np.arange(0,seconds,1/fs)
    clean=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
    if kind=='normal': x=clean+local.normal(0,.18,len(t))
    elif kind=='high_noise': x=clean+local.normal(0,.75,len(t))
    elif kind=='spike':
        x=clean+local.normal(0,.18,len(t)); idx=local.choice(len(t),4,False); x[idx]+=local.choice([-5,5],4)
    else: raise ValueError('kind 확인')
    return t,x

def build_dataset(n=40,fs=50):
    rows=[]; labels=['normal','high_noise','spike']
    for j,label in enumerate(labels):
        for i in range(n):
            _,x=generate_signal(label,fs=fs,seed=1000+100*j+i)
            row=extract_features(x,fs); row['label']=label; rows.append(row)
    return pd.DataFrame(rows)
print('준비 완료')


## 1. FFT로 3 Hz와 7 Hz 찾기


In [ ]:
fs=50;t=np.arange(0,4,1/fs);x=np.sin(2*np.pi*3*t)+.35*np.sin(2*np.pi*7*t)
freq=np.fft.rfftfreq(len(x),1/fs);spec=np.abs(np.fft.rfft(x-x.mean()))
plt.plot(freq,spec);plt.xlim(0,15);plt.xlabel('Hz');plt.show()
print('지배주파수:',freq[np.argmax(spec)])


**관찰 1** 가장 큰 피크와 두 번째 피크의 위치를 찾으세요.


## 2. 주기신호와 잡음 스펙트럼


In [ ]:
noise=rng.normal(0,1,len(t));sn=np.abs(np.fft.rfft(noise))
fig,ax=plt.subplots(1,2,figsize=(11,4));ax[0].plot(freq,spec);ax[0].set_xlim(0,15);ax[1].plot(freq,sn,color='gray');ax[1].set_xlim(0,15);plt.show()


**관찰 2** 피크가 집중된 신호와 넓게 퍼진 잡음을 비교하세요.


## 3. PSD와 AI 주파수 특징


In [ ]:
from scipy.signal import welch
f1,p1=welch(x,fs=fs,nperseg=100);f2,p2=welch(noise,fs=fs,nperseg=100)
plt.semilogy(f1,p1,label='periodic');plt.semilogy(f2,p2,label='noise');plt.xlim(0,15);plt.legend();plt.show()
for kind in ['normal','high_noise','spike']:
    _,sig=generate_signal(kind,fs,seed=20);f=extract_features(sig,fs)
    print(kind,round(f['dominant_frequency'],2),round(f['spectral_energy'],2))


**미니 과제** 세 상태의 지배주파수와 스펙트럼 에너지를 비교하세요.


## 참고 자료
- NumPy rfft: https://numpy.org/doc/stable/reference/generated/numpy.fft.rfft.html
- NumPy rfftfreq: https://numpy.org/doc/stable/reference/generated/numpy.fft.rfftfreq.html
- SciPy welch: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.welch.html
